In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 📊 Student Performance Prediction with Linear Regression

**🇹🇷 Türkçe:**
Bu projenin amacı, öğrencilerin çalışma saatleri, önceki sınav notları, uyku saatleri ve müfredat dışı etkinliklere katılım durumları gibi bağımsız değişkenleri kullanarak **Performans İndekslerini** tahmin etmektir. İlk adımda veri setimizi yüklüyor ve genel yapısını inceliyoruz.

**🇬🇧 English:**
The objective of this project is to predict the **Performance Index** of students based on independent variables such as hours studied, previous scores, sleep hours, and extracurricular activities. In the first step, we load our dataset and inspect its general structure.

In [2]:
df = pd.read_csv("../../Data/Student_Performance.csv")

In [3]:
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Hours Studied                     10000 non-null  int64  
 1   Previous Scores                   10000 non-null  int64  
 2   Extracurricular Activities        10000 non-null  str    
 3   Sleep Hours                       10000 non-null  int64  
 4   Sample Question Papers Practiced  10000 non-null  int64  
 5   Performance Index                 10000 non-null  float64
dtypes: float64(1), int64(4), str(1)
memory usage: 468.9 KB


In [5]:
df.describe()

,Hours Studied,Previous Scores,Sleep Hours,Sample Question Papers Practiced,Performance Index
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,4.992900,69.445700,6.530600,4.583300,55.224800
std,2.589309,17.343152,1.695863,2.867348,19.212558
min,1.000000,40.000000,4.000000,0.000000,10.000000
25%,3.000000,54.000000,5.000000,2.000000,40.000000
50%,5.000000,69.000000,7.000000,5.000000,55.000000
75%,7.000000,85.000000,8.000000,7.000000,71.000000
max,9.000000,99.000000,9.000000,9.000000,100.000000


In [6]:
df.isnull().sum()

Hours Studied                       0
Previous Scores                     0
Extracurricular Activities          0
Sleep Hours                         0
Sample Question Papers Practiced    0
Performance Index                   0
dtype: int64

## ⚙️ Veri Ön İşleme ve Özellik Mühendisliği / Data Preprocessing & Feature Engineering

**🇹🇷 Türkçe:**
Makine öğrenmesi modelleri kategorik metinleri doğrudan işleyemez. Bu bölümde:
1. Veriyi Eğitim (Train) ve Test setlerine ayırıyoruz.
2. `Extracurricular Activities` (Evet/Hayır) sütununu modelin anlayabileceği (1/0) formatına çeviriyoruz.
3. Farklı ölçeklerdeki sayısal verilerin modelde baskınlık kurmasını engellemek için **Standardizasyon (Z-Score)** uyguluyoruz.

**🇬🇧 English:**
Machine learning models cannot process categorical text directly. In this section:
1. We split the data into Training and Test sets.
2. We encode the `Extracurricular Activities` (Yes/No) column into a machine-readable format (1/0).
3. We apply **Standardization (Z-Score)** to prevent numerical features with different scales from dominating the model.

In [14]:
from sklearn.model_selection import train_test_split

In [19]:
X = df[["Hours Studied","Previous Scores","Extracurricular Activities", "Sleep Hours", "Sample Question Papers Practiced"]]
y = df["Performance Index"]

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 15)

In [23]:
X_train["Extracurricular_encoded"] = X_train["Extracurricular Activities"].map({"No": 0, "Yes": 1})
X_test["Extracurricular_encoded"] = X_test["Extracurricular Activities"].map({"No": 0, "Yes": 1})

In [24]:
X_train

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Extracurricular_encoded
3190,7,67,No,8,0,0
5091,3,81,No,8,8,0
3841,7,90,No,9,8,0
1608,2,97,No,5,9,0
9364,8,84,No,9,2,0
...,...,...,...,...,...,...
6528,7,40,No,9,3,0
2693,6,79,No,6,9,0
8076,3,92,No,4,4,0
3829,9,67,Yes,5,7,1


In [25]:
X_test

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Extracurricular_encoded
522,6,58,Yes,8,7,1
5667,9,49,No,5,0,0
4030,6,89,No,9,8,0
3198,2,64,No,8,0,0
2902,1,53,Yes,6,3,1
...,...,...,...,...,...,...
9855,5,83,No,7,2,0
9040,2,62,No,8,9,0
128,1,97,Yes,8,0,1
3465,6,80,Yes,8,3,1


In [26]:
X_test.drop(["Extracurricular Activities"], axis=1, inplace=True)

In [27]:
X_train.drop(["Extracurricular Activities"], axis=1, inplace=True)

In [28]:
X_train

,Hours Studied,Previous Scores,Sleep Hours,Sample Question Papers Practiced,Extracurricular_encoded
3190,7,67,8,0,0
5091,3,81,8,8,0
3841,7,90,9,8,0
1608,2,97,5,9,0
9364,8,84,9,2,0
...,...,...,...,...,...
6528,7,40,9,3,0
2693,6,79,6,9,0
8076,3,92,4,4,0
3829,9,67,5,7,1


In [29]:
y_train

3190    56.0
5091    65.0
3841    82.0
1608    73.0
9364    81.0
        ... 
6528    30.0
2693    68.0
8076    70.0
3829    64.0
7624    47.0
Name: Performance Index, Length: 7500, dtype: float64

In [30]:
from sklearn.preprocessing import StandardScaler

In [31]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [32]:
X_train

array([[ 0.7725075 , -0.13738686,  0.85938628, -1.62071899, -0.98965352],
       [-0.77291962,  0.67136016,  0.85938628,  1.19474004, -0.98965352],
       [ 0.7725075 ,  1.19126896,  1.4517941 ,  1.19474004, -0.98965352],
       ...,
       [-0.77291962,  1.30680425, -1.51024501, -0.21298947, -0.98965352],
       [ 1.54522106, -0.13738686, -0.91783718,  0.84280766,  1.01045465],
       [ 1.54522106, -0.88836624,  0.85938628, -0.91685423, -0.98965352]],
      shape=(7500, 5))

## 🤖 Model Eğitimi / Model Training

**🇹🇷 Türkçe:**
Verilerimiz hazır olduğuna göre, Çoklu Doğrusal Regresyon (Multiple Linear Regression) modelimizi kurabilir ve eğitim verilerimiz (`X_train`, `y_train`) ile modelimizi eğitebiliriz.
Denklem yapısı: $y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + ... + \epsilon$

**🇬🇧 English:**
Now that our data is ready, we can build our Multiple Linear Regression model and fit it using our training data (`X_train`, `y_train`).
Equation structure: $y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + ... + \epsilon$

In [33]:
from sklearn.linear_model import LinearRegression

In [34]:
regression = LinearRegression()

In [35]:
regression.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](5,)","[ 7.38,17.62, 0.82, 0.56, 0.3 ]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,55.17
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(5)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](5,)","[88.04,87.63,87.16,85.51,84.63]"


In [36]:
1df

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0
...,...,...,...,...,...,...
9995,1,49,Yes,4,2,23.0
9996,7,64,Yes,8,5,58.0
9997,6,83,Yes,8,5,74.0
9998,9,97,Yes,7,0,95.0


In [45]:
new_student = [[7,99,1,9,1]]

In [46]:
new_student_scaled = scaler.transform(new_student)

C:\Users\efeka\PycharmProjects\PythonProject1\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [47]:
regression.predict(new_student_scaled)

array([89.47090241])

## 📈 Model Değerlendirme ve Sonuçlar / Model Evaluation & Results

**🇹🇷 Türkçe:**
Eğittiğimiz modelin test verisi üzerindeki performansını ölçmek için regresyon metriklerini kullanıyoruz:
* **MAE (Mean Absolute Error):** Ortalama mutlak hata.
* **MSE (Mean Squared Error):** Ortalama karesel hata.
* **$R^2$ Skoru:** Bağımsız değişkenlerin, hedef değişkendeki varyansı ne kadar iyi açıkladığının yüzdesidir. 1.0'a ne kadar yakınsa o kadar iyidir.

**🇬🇧 English:**
We use regression metrics to evaluate the performance of our trained model on the test data:
* **MAE (Mean Absolute Error):** Average absolute differences between predicted and actual values.
* **MSE (Mean Squared Error):** Average squared differences.
* **$R^2$ Score:** Represents the proportion of variance for the dependent variable that's explained by independent variables. Closer to 1.0 is better.

In [48]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [50]:
y_pred = regression.predict(X_test)

In [51]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [52]:
print(mae)
print(mse)
print(r2)

1.6341484613506643
4.256407547565782
0.9886098260837627
